# SROIE task-field removal sensitivity
Exploratory annotation-matched subsets. Not a PII-category or cross-domain policy superiority test. All 350 baselines are rerun; only unique exact transcript matches are masked. Ambiguous and unmatched cases are excluded before inference. Address subset is small and selected. No target-field recovery expected to remain perfect; other visible duplicate values and layout may still allow inference. Company/address are receipt fields, not necessarily personal identifiers. Date/total are not automatically PII. Use existing Drive SROIE_Current_C2 folder; add SROIE_TaskField_Manifest.json. Source ZIP and full metadata remain required.

In [ ]:
!pip -q install "transformers>=4.46,<5" sentencepiece accelerate pandas scipy
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import io,json,zipfile,re,hashlib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import DonutProcessor,VisionEncoderDecoderModel
BASE=Path('/content/drive/MyDrive/SROIE_Current_C2')
ORIGINAL=BASE/'SROIE-PII-full971-images-test.zip'
MASKED=BASE/'SROIE_Current_C2_Rendered350.zip'
METADATA=BASE/'SROIE-PII-full971-metadata.zip'
OUT=BASE/'TaskField_Results';OUT.mkdir(exist_ok=True)
assert torch.cuda.is_available(),'Select GPU runtime'
with zipfile.ZipFile(METADATA) as z:docs=[d for d in json.loads(z.read('sroie_pii_full971.json'))['documents'] if d['split']=='test']
assert len(docs)==350
MODEL='philschmid/donut-base-sroie'
REVISION='51e72e5d160efa472ee3958d79a766faa42f6c5d'
processor=DonutProcessor.from_pretrained(MODEL,revision=REVISION)
model=VisionEncoderDecoderModel.from_pretrained(MODEL,revision=REVISION).to('cuda').eval()
revision=getattr(model.config,'_commit_hash',None)
prompt=processor.tokenizer('<s>',add_special_tokens=False,return_tensors='pt').input_ids.to('cuda')
fields=['company','date','address','total']
def norm(s):return re.sub('[^a-z0-9]','',str(s or '').casefold())

SOURCE=BASE/'SROIE-original-test.zip'
assert SOURCE.exists(), 'Add the genuine unaugmented original image ZIP; do not rename the augmented ZIP.'

# Verify exact source provenance before inference. Filenames may be nested.
source_names={}
with zipfile.ZipFile(SOURCE) as z:
 for n in z.namelist():
  if n.lower().endswith(('.jpg','.jpeg','.png')):
   key=Path(n).stem
   if key in source_names:raise ValueError('Duplicate image ID: '+key)
   source_names[key]=n
 checks=[]
 for d in docs:
  key=d['document_id'];assert key in source_names, 'Missing original: '+key
  data=z.read(source_names[key]);sha=hashlib.sha256(data).hexdigest()
  assert sha==d['source_image_sha256'], 'Original hash mismatch: '+key
  im=Image.open(io.BytesIO(data));assert im.size==(d['width'],d['height'])
  checks.append({'document_id':key,'sha256':sha,'matches_source':True})
pd.DataFrame(checks).to_csv(OUT/'source_checks.csv',index=False)

MANIFEST=BASE/'SROIE_TaskField_Manifest.json'
spec=json.loads(MANIFEST.read_text())
assert len(spec['entries'])==1400
eligible={(e['document_id'],e['field']):e for e in spec['entries'] if e['status']=='eligible'}
assert all(e['source_image_sha256']==next(d['source_image_sha256'] for d in docs if d['document_id']==e['document_id']) for e in spec['entries'])
(OUT/'manifest_used.json').write_text(json.dumps(spec,indent=2))
from PIL import ImageDraw


In [ ]:

checkpoint=OUT/'checkpoint_predictions.csv'
run={'revision':REVISION,'source_sha256':hashlib.sha256(SOURCE.read_bytes()).hexdigest(),'metadata_sha256':hashlib.sha256(METADATA.read_bytes()).hexdigest(),'manifest_sha256':hashlib.sha256(MANIFEST.read_bytes()).hexdigest()}
runpath=OUT/'run_spec.json'
if runpath.exists():assert json.loads(runpath.read_text())==run,'Changed inputs; use a fresh output folder.'
else:runpath.write_text(json.dumps(run,indent=2))
records=pd.read_csv(checkpoint,keep_default_na=False).to_dict('records') if checkpoint.exists() else []
done={(r['document_id'],r['condition']) for r in records};assert len(done)==len(records)
total=350+len(eligible)
with zipfile.ZipFile(SOURCE) as zs:
 for d in docs:
  conditions=['SOURCE_UNAUGMENTED']+['MASK_'+f for f in fields if (d['document_id'],f) in eligible]
  for condition in conditions:
   if (d['document_id'],condition) in done:continue
   im=Image.open(io.BytesIO(zs.read(source_names[d['document_id']]))).convert('RGB')
   if condition!='SOURCE_UNAUGMENTED':
    e=eligible[d['document_id'],condition[5:]]
    before=np.asarray(im).copy();union=np.zeros((im.height,im.width),dtype=bool)
    for x0,y0,x1,y1 in e['boxes']:union[y0:y1,x0:x1]=True
    after=before.copy();after[union]=0;im=Image.fromarray(after)
    assert np.array_equal(after[~union],before[~union]) and np.all(after[union]==0)
   pixels=processor(im,return_tensors='pt').pixel_values.to('cuda')
   with torch.inference_mode():
    seq=model.generate(pixels,decoder_input_ids=prompt,max_length=model.decoder.config.max_position_embeddings,early_stopping=True,pad_token_id=processor.tokenizer.pad_token_id,eos_token_id=processor.tokenizer.eos_token_id,use_cache=True,num_beams=1,bad_words_ids=[[processor.tokenizer.unk_token_id]])
   raw=processor.batch_decode(seq)[0]
   cleaned=raw.replace(processor.tokenizer.eos_token or '', '').replace(processor.tokenizer.pad_token or '', '').strip()
   try:pred=processor.token2json(cleaned)
   except Exception:pred={}
   if not isinstance(pred,dict):pred={}
   r={'document_id':d['document_id'],'condition':condition,'raw_prediction':raw}
   exact=[]
   for f in fields:
    gt=d['source_entities'][f];pv=pred.get(f,'')
    r['gt_'+f]=gt;r['pred_'+f]=str(pv);r['exact_'+f]=int(norm(gt)==norm(pv));exact.append(r['exact_'+f])
   r['field_exact']=float(np.mean(exact));records.append(r)
   pd.DataFrame(records).to_csv(OUT/'checkpoint_predictions.csv',index=False)
   if len(records)%25==0:print(len(records),'/',total)


In [ ]:

df=pd.DataFrame(records);assert len(df)==total and not df.duplicated(['document_id','condition']).any()
df.to_csv(OUT/'predictions.csv',index=False)
base=df[df.condition=='SOURCE_UNAUGMENTED'].set_index('document_id')
results=[]
for f in fields:
 masked=df[df.condition=='MASK_'+f].set_index('document_id')
 n=len(masked)
 if not n:continue
 ref=base.loc[masked.index]
 for outcome in ['target_field','other_fields','overall']:
  cols=['exact_'+f] if outcome=='target_field' else (['exact_'+x for x in fields if x!=f] if outcome=='other_fields' else ['exact_'+x for x in fields])
  a=ref[cols].astype(float).mean(axis=1).to_numpy();b=masked[cols].astype(float).mean(axis=1).to_numpy();delta=b-a
  rng=np.random.default_rng(20261003);boot=[rng.choice(delta,n,replace=True).mean() for _ in range(10000)];lo,hi=np.quantile(boot,[.025,.975])
  results.append(dict(masked_field=f,outcome=outcome,documents=n,baseline=float(a.mean()),masked=float(b.mean()),difference=float(delta.mean()),ci95_low=float(lo),ci95_high=float(hi)))
pd.DataFrame(results).to_csv(OUT/'ablation_summary.csv',index=False)
print(pd.DataFrame(results));print('Return ablation_summary.csv, predictions.csv, manifest_used.json and run_spec.json.')
